# 🎲 Fate Core Сonsultor: Семантический анализ базы знаний

**Исследование векторного пространства правил настольной игры**

1. Цель проекта:

Создать фундамент для RAG-системы (Retrieval-Augmented Generation), проанализировав, как нейросеть «понимает» структуру и смысл правил настольной ролевой игры Fate Core. Мы проверяем, насколько качественно модель эмбеддингов группирует игровые механики и примеры.

2. Методы и стек:

- Data Ingestion: Программная загрузка актуального датасета из официального GitHub-репозитория.
- NLP & Embeddings: Использование модели all-MiniLM-L6-v2 для превращения текста в 384-мерные векторы.
- Dimensionality Reduction: Применение алгоритма t-SNE для визуализации сложной структуры данных на 2D-плоскости.
- Clustering: Автоматическая сегментация тем с помощью K-Means и анализ ключевых слов через TF-IDF.

3. Зачем это нужно?

Прежде чем строить чат-бота, важно убедиться, что система поиска (Retriever) работает корректно. Этот ноутбук — «рентгеновский снимок» того, как ИИ структурирует информацию, разделяя сухие правила и художественные примеры игры.
 

In [2]:
!pip install -q sentence-transformers huggingface_hub

## 🛠 Шаг 1. Инициализация окружения

In [3]:
import os
import re
import numpy as np
import pandas as pd
import plotly.express as px
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from IPython.display import display, Markdown

- Модели: Загружаем предобученные модели эмбеддингов для семантического анализа.
- Данные: В качестве базы знаний используем полный текст правил настольной ролевой игры Fate Core, загруженный напрямую из официального репозитория.


In [4]:
# 1. ЗАГРУЗКА МОДЕЛИ И ПОДГОТОВКА ДАННЫХ
print("📦 Загрузка модели эмбеддингов (all-MiniLM-L6-v2)...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

📦 Загрузка модели эмбеддингов (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Функция для очистки текста 
def smart_clean(text):
    """Очистка текста с сохранением структуры Markdown."""
    # Удаляем ссылки [text](url) -> оставляем только text
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    # Удаляем лишние технические линии (---)
    text = re.sub(r'-{3,}', '', text)
    # Нормализуем пробелы и пустые строки
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

In [6]:
# Скачиваем файл напрямую из Raw-репозитория GitHub
url = 'https://raw.githubusercontent.com/fate-srd/fate-srd-content/main/docs/markdown/fate-core.md'
!wget {url} -O fate-core.md

# Проверяем, что файл скачался и он не пустой
import os
if os.path.exists("fate-core.md"):
    size = os.path.getsize("fate-core.md")
    print(f"✅ Файл успешно скачан! Размер: {size / 1024:.2f} КБ")
else:
    print("❌ Файл не найден. Проверьте настройки интернета в Kaggle (правая панель).")

--2026-04-24 10:25:24--  https://raw.githubusercontent.com/fate-srd/fate-srd-content/main/docs/markdown/fate-core.md
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 494358 (483K) [text/plain]
Saving to: ‘fate-core.md’

fate-core.md        100%[===================>] 482.77K  --.-KB/s    in 0.01s   

2026-04-24 10:25:24 (49.1 MB/s) - ‘fate-core.md’ saved [494358/494358]

✅ Файл успешно скачан! Размер: 482.77 КБ


In [7]:
# Загружаем файл, который мы скачали ранее
file_path = "fate-core.md"
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

- окружение подготовлено

## 🧩 Шаг 2. Препроцессинг и создание векторного индекса
- Chunking (Нарезка): Разбиваем текст на смысловые фрагменты (чанки) с сохранением Markdown-разметки. Это позволяет модели учитывать структуру заголовков и акценты на важных терминах.
- Векторизация (Embeddings): Превращаем каждый текстовый фрагмент в многомерный вектор (384 измерения). В таком виде правила становятся понятны нейросети для поиска по смыслу.
- Валидация: Проводим инспекцию полученных чанков, чтобы убедиться в чистоте данных перед анализом.

In [8]:
# Нарезка на чанки по двойному переносу строки
raw_chunks = [c.strip() for c in raw_text.split("\n\n") if len(c.strip()) > 60]
CHUNKS = [smart_clean(c) for c in raw_chunks]

In [9]:
# 2. ВЫВОД ПРИМЕРОВ (EDA)
print(f"\n✅ Всего подготовлено качественных чанков: {len(CHUNKS)}")
print("-" * 60)
# Выведем 3 характерных примера (начало, середина, конец)
indices = [10, len(CHUNKS)//2, len(CHUNKS)-50]
for i in indices:
    print(f"📍 ПРИМЕР ЧАНКА №{i}:")
    print(f"{CHUNKS[i][:400]}...") # Показываем первые 400 символов
    print("-" * 60)


✅ Всего подготовлено качественных чанков: 1632
------------------------------------------------------------
📍 ПРИМЕР ЧАНКА №10:
### What You Need to Play
Getting into a game of Fate is very simple. You need:...
------------------------------------------------------------
📍 ПРИМЕР ЧАНКА №816:
##### Resolving Attacks
A successful attack lands a hit equivalent to its shift value on a target. So if you get three shifts on an attack, you land a 3-shift hit....
------------------------------------------------------------
📍 ПРИМЕР ЧАНКА №1582:
###### Sujan, the Spirit of Warding
**Portfolio**: Defense and protection...
------------------------------------------------------------


- Качество контента: Визуальная инспекция подтвердила чистоту данных. Дополнительная фильтрация шума на данном этапе не требуется.
- Сохранение структуры: Было принято решение сохранить Markdown-разметку (#, **). Это позволит модели использовать внутреннюю иерархию документа и улучшить акцентирование внимания на ключевых терминах через встроенные механизмы Attention.
- Готовность к векторизации: Выбранная стратегия нарезки (chunking) обеспечила оптимальный баланс между сохранением контекста и размером вектора.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 3. ВЕКТОРИЗАЦИЯ
print("\n💎 Превращаем правила в векторы (эмбеддинги)...")
embeddings = model.encode(CHUNKS, show_progress_bar=True)

# === 4. КЛАСТЕРИЗАЦИЯ И ГЕНЕРАЦИЯ НАЗВАНИЙ ТЕМ ===
print("🤖 Группировка тем и анализ ключевых слов...")

num_clusters = 6
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(embeddings)

# Группируем тексты по кластерам для анализа TF-IDF
cluster_docs = ["" for _ in range(num_clusters)]
for i, label in enumerate(clusters):
    cluster_docs[label] += " " + CHUNKS[i]

# Настраиваем TF-IDF, чтобы найти уникальные игровые термины
# Игнорируем цифры и общие слова (stop_words)
vectorizer = TfidfVectorizer(
    stop_words='english', 
    token_pattern=r'[a-zA-Z]{4,}', # слова от 4 букв (убираем мусор и предлоги)
    max_features=500
)

tfidf_matrix = vectorizer.fit_transform(cluster_docs)
feature_names = vectorizer.get_feature_names_out()

# Генерируем емкие названия: берем топ-3 слова для каждого кластера
cluster_labels_map = {}
for i in range(num_clusters):
    top_indices = tfidf_matrix[i].toarray().argsort()[0][-3:][::-1]
    top_words = [feature_names[idx].upper() for idx in top_indices]
    cluster_labels_map[i] = f"ТЕМА: {' | '.join(top_words)}"



💎 Превращаем правила в векторы (эмбеддинги)...


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

🤖 Группировка тем и анализ ключевых слов...


- данные подготовлены

## 📊 Шаг 3. Семантическая кластеризация и визуализация

- Задача: Выявить скрытую структуру в правилах игры и проверить, насколько кучно группируются схожие игровые механики.
- Метод: Используем алгоритм t-SNE для проекции многомерных векторов на плоскость и K-Means для автоматического сегментирования тем.
- Анализ: Оцениваем плотность распределения данных и корректность разделения контента на «правила» и «примеры».

In [14]:
# === 5. ВИЗУАЛИЗАЦИЯ ===
print("🗺 Строим интерактивную карту...")

# понижение размерности
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
vis_dims = tsne.fit_transform(embeddings)

# упаковка данных
df_vis = pd.DataFrame({
    'x': vis_dims[:, 0],
    'y': vis_dims[:, 1],
    'topic': [cluster_labels_map[c] for c in clusters],
    'preview': [t[:250] + "..." for t in CHUNKS]
})

fig = px.scatter(
    df_vis, x='x', y='y', color='topic',
    hover_data={'preview': True, 'x': False, 'y': False, 'topic': False},
    title="<b>Семантическая карта правил Fate Core</b><br><sup>Названия тем определены автоматически через анализ частоты терминов</sup>",
    template="plotly_dark",
    color_discrete_sequence=px.colors.qualitative.Bold,
    labels={'topic': 'Направление правил'}
)

fig.update_traces(marker=dict(size=10, opacity=0.8, line=dict(width=1, color='white')))
fig.update_xaxes(showticklabels=False, title="")
fig.update_yaxes(showticklabels=False, title="")

fig.show()

# Печатаем расшифровку тем для проверки
print("\n📝 РАСШИФРОВКА ТЕМ:")
for i, name in cluster_labels_map.items():
    print(f"{i+1}. {name}")

🗺 Строим интерактивную карту...



📝 РАСШИФРОВКА ТЕМ:
1. ТЕМА: ATTACK | DEFEND | ROLL
2. ТЕМА: GAME | CHARACTER | TIME
3. ТЕМА: ASPECTS | ASPECT | CHARACTER
4. ТЕМА: CYNERE | AMANDA | ZIRD
5. ТЕМА: SKILL | SKILLS | CHARACTER
6. ТЕМА: FATE | CHARACTER | GAME


In [13]:
# === ПОДГОТОВКА КОМПАКТНЫХ МЕТОК ДЛЯ ЛЕГЕНДЫ ===

# Создаем словарь: {индекс_кластера: "1. СЛОВО"}
short_labels_map = {}
for i in range(num_clusters):
    # Берем первое слово из нашего словаря названий (оно там идет первым после двоеточия)
    first_keyword = cluster_labels_map[i].split(': ')[1].split(' | ')[0]
    short_labels_map[i] = f"{i+1}. {first_keyword}"

# === ДЕТАЛЬНЫЙ ГРАФИК ===

df_compact = pd.DataFrame({
    'x': vis_dims[:, 0],
    'y': vis_dims[:, 1],
    'topic_short': [short_labels_map[c] for c in clusters], # Короткая метка для легенды
    'keywords_full': [cluster_labels_map[c] for c in clusters], # Полный список для тултипа
    'preview': [t[:250] + "..." for t in CHUNKS]
})

fig_compact = px.scatter(
    df_compact, x='x', y='y', color='topic_short',
    hover_data={'preview': True, 'keywords_full': True, 'topic_short': False, 'x': False, 'y': False},
    title="<b>Семантическая карта Fate Core</b>",
    template="plotly_dark",
    color_discrete_sequence=px.colors.qualitative.Prism,
    width=1000, 
    height=750
)

# Настройка стиля
fig_compact.update_traces(marker=dict(size=7, opacity=0.8, line=dict(width=0.5, color='white')))

# Горизонтальная легенда сверху с короткими названиями
fig_compact.update_layout(
    margin=dict(l=10, r=10, t=100, b=10),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        title_text="" 
    )
)

fig_compact.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, title="")
fig_compact.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, title="")

fig_compact.show()

# Расшифровка под графиком остается для полной информации
display(Markdown("### 📚 Полная расшифровка тем:"))
summary = [f"**{short_labels_map[i]}**: {cluster_labels_map[i].split(': ')[1]}" for i in range(num_clusters)]
display(Markdown("\n\n".join(summary)))

### 📚 Полная расшифровка тем:

**1. ATTACK**: ATTACK | DEFEND | ROLL

**2. GAME**: GAME | CHARACTER | TIME

**3. ASPECTS**: ASPECTS | ASPECT | CHARACTER

**4. CYNERE**: CYNERE | AMANDA | ZIRD

**5. SKILL**: SKILL | SKILLS | CHARACTER

**6. FATE**: FATE | CHARACTER | GAME

- На кластеризации проявилась тема  'CYNERE, AMANDA, ZIRD' - сгруппировались примеры из книги правил с персонажами и игроками. В целом это показыват, что модель отработала хорошо. 
- Модель эмбеддингов (MiniLM) оказалась настолько чувствительной, что смогла уловить разницу в тональности между инструктивным текстом (правила) и нарративным текстом (примеры сессий). Это подтверждает высокое качество векторных представлений.

In [15]:
import plotly.graph_objects as go

# 1. Считаем количество чанков в каждой группе
counts = pd.Series(clusters).value_counts().sort_index()
topic_names = [short_labels_map[i] for i in range(num_clusters)]

# 2. Строим столбчатую диаграмму
fig_bar = go.Figure(data=[
    go.Bar(
        x=topic_names,
        y=counts,
        text=counts, # Выводим цифры прямо над столбцами
        textposition='auto',
        marker_color=px.colors.qualitative.Prism[:num_clusters], # Те же цвета, что на карте
        opacity=0.8
    )
])

fig_bar.update_layout(
    title="<b>Распределение объема знаний по темам</b><br><sup>Количество фрагментов правил в каждом кластере</sup>",
    xaxis_title="Тематическая группа",
    yaxis_title="Количество чанков",
    template="plotly_dark",
    margin=dict(l=20, r=20, t=80, b=20),
    height=500
)

fig_bar.show()

# 3. Краткий аналитический комментарий (для Markdown ячейки)
imbalance_ratio = max(counts) / min(counts)
display(Markdown(f"**Аналитическая заметка:** Дисбаланс между самой крупной и мелкой группой составляет всего {imbalance_ratio:.2f}x. Это говорит о достаточно равномерном покрытии правил в базе знаний."))

**Аналитическая заметка:** Дисбаланс между самой крупной и мелкой группой составляет всего 2.61x. Это говорит о достаточно равномерном покрытии правил в базе знаний.

- В целом данные распределены без экстремальных перекосов, что хоршо для модели интерпретатора. По каждому топику у нее будет достаточно примеров в разных сценариях.   

## 🏁 Общие выводы

- По итогам семантического анализа правил Fate Core можно сделать следующие заключения:

1. Качество векторного представления: Выбранная модель эмбеддингов (all-MiniLM-L6-v2) успешно справилась с задачей. На визуализации t-SNE четко видны сформированные кластеры, что подтверждает высокую семантическую плотность данных.

2. Структурная чистота: Нам удалось успешно разделить «нормативный» текст правил и «нарративные» примеры. Выделение имен персонажей (Cynere, Zird) в отдельную группу — это не ошибка, а доказательство того, что модель различает функциональные стили текста.

3. Равномерность базы знаний: Анализ распределения чанков по темам показал отсутствие критических перекосов. Это гарантирует, что будущая RAG-система будет одинаково компетентна как в вопросах создания персонажа, так и в механике конфликтов.

4. Готовность к инференсу: Сохраненная Markdown-разметка и продуманная нарезка текста (chunking) создают идеальный фундамент для подключения LLM. Система поиска (Retriever) готова выдавать точный контекст для генерации ответов.

**Итог: Исследование подтвердило, что база данных Fate Core пригодна для развертывания интеллектуального ассистента.** Мы получили прозрачную, предсказуемую и масштабируемую структуру знаний.